In [10]:
# ============================================================
# Hospital Cyberattack SI Propagation — Interactive Map
# Run in VS Code:  python hospital_si_interactive.py
# Output:          hospital_si_map.html  (opens in browser)
#
# Files required in same folder as this script:
#   si_hospitals.json
#   si_edges.json
#   si_icb_boundaries.geojson
# ============================================================

In [11]:
import json
import math
import os
import webbrowser
import pandas as pd
import numpy as np

# ── 1. LOAD DATA

In [12]:

with open("si_hospitals.json")       as f: hospitals_raw = json.load(f)
with open("si_edges.json")           as f: edges_raw     = json.load(f)
with open("si_icb_boundaries.geojson") as f: icb_geo     = json.load(f)

# ── 2. BUILD ADJACENCY for SI model

In [13]:
# Pre-compute for each hospital: list of (neighbour_id, weight)

max_weight = max(e["weight"] for e in edges_raw) if edges_raw else 1

adjacency = {}   # hospital_id -> list of {neighbour, weight, prob}
for h in hospitals_raw:
    adjacency[h["id"]] = []

BETA = 0.5  # base transmission rate

for edge in edges_raw:
    s, t, w = edge["source"], edge["target"], edge["weight"]
    if s not in adjacency: adjacency[s] = []
    if t not in adjacency: adjacency[t] = []
    prob = 1 - math.exp(-BETA * (w / max_weight))
    adjacency[s].append({"neighbour": t, "weight": w, "prob": round(prob, 5)})
    adjacency[t].append({"neighbour": s, "weight": w, "prob": round(prob * 0.6, 5)})

# ── 3. PRE-RUN SI SIMULATIONS (N_RUNS per seed hospital)

In [14]:
# For each hospital as a single seed, run N_RUNS stochastic SI
# simulations and record which step each hospital got infected.
# This is stored as a lookup table so the browser app is instant.

N_RUNS      = 200   # simulations per seed hospital
MAX_STEPS   = 60

print(f"Pre-computing SI simulations ({len(hospitals_raw)} seeds × {N_RUNS} runs)...")
print("This may take 1–2 minutes...")

# hospital_id -> list of N_RUNS dicts: {step: [list of newly infected ids]}
# We store per-run infection timeline compactly as:
#   { hospital_id: median_step_infected }  across all runs
# Full format: seed_id -> { hospital_id: { median_step, freq } }

simulation_results = {}  # seed_id -> { hosp_id: {median_step, freq} }

hospital_ids = [h["id"] for h in hospitals_raw]

for seed_idx, seed_id in enumerate(hospital_ids):
    if seed_idx % 25 == 0:
        print(f"  Simulating seed {seed_idx+1}/{len(hospital_ids)}: {seed_id}")

    infection_steps = {h: [] for h in hospital_ids}  # hosp -> list of steps when infected

    for run in range(N_RUNS):
        infected = {seed_id}
        step_infected = {seed_id: 0}

        for step in range(1, MAX_STEPS + 1):
            newly = []
            for inf_node in list(infected):
                for edge in adjacency.get(inf_node, []):
                    nb = edge["neighbour"]
                    if nb not in infected:
                        if np.random.random() < edge["prob"]:
                            infected.add(nb)
                            newly.append(nb)
                            step_infected[nb] = step

            if not newly:
                break

        for h in hospital_ids:
            if h in step_infected:
                infection_steps[h].append(step_infected[h])
            # else: not infected in this run (stays as empty list)

    # Summarise across runs
    result = {}
    for h in hospital_ids:
        steps = infection_steps[h]
        freq  = len(steps) / N_RUNS
        result[h] = {
            "freq":         round(freq, 3),
            "median_step":  round(float(np.median(steps)), 1) if steps else None,
            "min_step":     int(min(steps))                   if steps else None,
            "max_step":     int(max(steps))                   if steps else None,
        }
    simulation_results[seed_id] = result

print("✓ Simulations complete.")

Pre-computing SI simulations (125 seeds × 200 runs)...
This may take 1–2 minutes...
  Simulating seed 1/125: RCF
  Simulating seed 26/125: RXR
  Simulating seed 51/125: RWF
  Simulating seed 76/125: RAL
  Simulating seed 101/125: RA9
✓ Simulations complete.


# ── 4. BUILD STEP-BY-STEP TIMELINE per seed

In [15]:
# For the animation: given seed S, at each step T, which hospitals
# have median_step_infected <= T?

timelines = {}  # seed_id -> { step: [list of hospital_ids infected by that step] }

for seed_id, result in simulation_results.items():
    timeline = {}
    for step in range(0, MAX_STEPS + 1):
        infected_by_step = []
        for h_id, stats in result.items():
            ms = stats["median_step"]
            if ms is not None and ms <= step:
                infected_by_step.append(h_id)
            elif h_id == seed_id:
                infected_by_step.append(h_id)
        timeline[step] = infected_by_step
    timelines[seed_id] = timeline

# ── 5. COMMUNITY COLOUR PALETTE

In [16]:
COMMUNITY_COLORS = [
    "#38bdf8", "#fb923c", "#4ade80", "#e879f9", "#facc15",
    "#f43f5e", "#06b6d4", "#a78bfa", "#34d399", "#fbbf24",
    "#60a5fa", "#f472b6"
]

community_ids = sorted(set(h["community_id"] for h in hospitals_raw))
color_map = {cid: COMMUNITY_COLORS[i % len(COMMUNITY_COLORS)] for i, cid in enumerate(community_ids)}

# ── 6. PREPARE DATA BLOBS FOR JAVASCRIPT

In [17]:
hospitals_js = []
for h in hospitals_raw:
    hospitals_js.append({
        "id":                   h["id"],
        "name":                 h["name"],
        "postcode":             h.get("postcode", ""),
        "lat":                  h["lat"],
        "lon":                  h["lon"],
        "community_id":         h["community_id"],
        "color":                color_map[h["community_id"]],
        "weighted_in_degree":   h.get("weighted_in_degree", 0),
        "total_transfers":      h.get("total_transfers", 0),
        "cross_community_bridges": h.get("cross_community_bridges", 0),
        "node_size":            h.get("node_size", 6),
        "infection_frequency":  h.get("infection_frequency"),
        "mean_epidemic_size_pct": h.get("mean_epidemic_size_pct"),
        "mean_step_to_infection": h.get("mean_step_to_infection"),
    })

edges_js = [
    {"source": e["source"], "target": e["target"], "weight": e["weight"]}
    for e in edges_raw
]

# Build hospital lookup for JS
hosp_lookup = {h["id"]: h for h in hospitals_js}

# Simulation results for JS: { seed: { hosp: {freq, median_step} } }
# Timelines for JS: { seed: { step: [ids] } }
# Convert step keys to int-string for JSON
timelines_js = {}
for seed, tl in timelines.items():
    timelines_js[seed] = {str(k): v for k, v in tl.items()}

# ── 7. GENERATE HTML

In [18]:
icb_geo_json = json.dumps(icb_geo)
hospitals_json = json.dumps(hospitals_js)
edges_json = json.dumps(edges_js)
sim_results_json = json.dumps(simulation_results)
timelines_json = json.dumps(timelines_js)
color_map_json = json.dumps(color_map)
community_ids_json = json.dumps(community_ids)

max_in_degree = max(h.get("weighted_in_degree", 1) for h in hospitals_js)
max_transfers = max(h.get("total_transfers", 1) for h in hospitals_js)

icb_geo_json = json.dumps(icb_geo)
hospitals_json = json.dumps(hospitals_js)
edges_json = json.dumps(edges_js)
sim_results_json = json.dumps(simulation_results)
timelines_json = json.dumps(timelines_js)
color_map_json = json.dumps(color_map)
community_ids_json = json.dumps(community_ids)

max_in_degree = max(h.get("weighted_in_degree", 1) for h in hospitals_js)
max_transfers = max(h.get("total_transfers", 1) for h in hospitals_js)

HTML = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>NHS Hospital Cyberattack SI Propagation</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ background:#f8fafc; color:#1e293b; font-family:'Courier New',monospace; display:flex; flex-direction:column; height:100vh; overflow:hidden; }}

  #header {{ background:#ffffff; border-bottom:1px solid #e2e8f0; padding:10px 18px; display:flex; align-items:center; justify-content:space-between; flex-shrink:0; box-shadow:0 1px 3px rgba(0,0,0,0.06); }}
  #header h1 {{ font-size:13px; letter-spacing:0.12em; color:#64748b; font-weight:normal; }}
  #status-bar {{ display:flex; gap:22px; font-size:11px; color:#94a3b8; }}
  #status-bar span {{ color:#1e293b; }}

  #main {{ display:flex; flex:1; overflow:hidden; }}

  /* LEFT PANEL */
  #left-panel {{ width:300px; border-right:1px solid rgba(255,255,255,0.07); display:flex; flex-direction:column; overflow:hidden; flex-shrink:0; }}
  .panel-section {{ padding:14px 16px; border-bottom:1px solid rgba(255,255,255,0.06); }}
  .panel-label {{ font-size:9px; letter-spacing:0.15em; color:#475569; margin-bottom:10px; }}

  #search-box {{ width:100%; background:rgba(255,255,255,0.05); border:1px solid rgba(255,255,255,0.1); border-radius:4px; padding:6px 10px; color:#e2e8f0; font-size:11px; font-family:inherit; outline:none; }}
  #hospital-list {{ max-height:200px; overflow-y:auto; margin-top:6px; }}
  .hosp-item {{ display:flex; align-items:center; gap:8px; padding:5px 6px; cursor:pointer; border-radius:3px; border:1px solid transparent; margin-bottom:2px; transition:all 0.15s; }}
  .hosp-item:hover {{ background:#f1f5f9; }}
  .hosp-item.selected {{ background:rgba(255,45,85,0.08); border-color:rgba(255,45,85,0.35); }}
  .hosp-dot {{ width:8px; height:8px; border-radius:50%; flex-shrink:0; }}
  .hosp-item-text {{ flex:1; min-width:0; }}
  .hosp-id {{ font-size:10px; color:#1e293b; }}
  .hosp-name {{ font-size:8px; color:#94a3b8; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }}
  .hosp-risk {{ font-size:9px; color:#94a3b8; }}

  #seeds-display {{ display:flex; flex-wrap:wrap; gap:4px; min-height:22px; }}
  .seed-tag {{ font-size:9px; padding:2px 7px; border-radius:2px; background:rgba(255,45,85,0.1); border:1px solid rgba(255,45,85,0.35); color:#e11d48; cursor:pointer; }}

  #btn-row {{ display:flex; gap:6px; margin-top:10px; }}
  .btn {{ flex:1; padding:7px 0; font-size:10px; letter-spacing:0.1em; border:none; border-radius:3px; cursor:pointer; font-family:inherit; }}
  #btn-run {{ background:#e11d48; color:white; }}
  #btn-run:disabled {{ background:#e2e8f0; color:#94a3b8; cursor:not-allowed; }}
  #btn-reset {{ background:#f1f5f9; border:1px solid #e2e8f0; color:#64748b; }}
  #btn-export {{ background:rgba(59,130,246,0.08); border:1px solid rgba(59,130,246,0.25); color:#3b82f6; }}

  /* Playback controls */
  #playback {{ display:flex; align-items:center; gap:8px; margin-top:8px; }}
  #step-slider {{ flex:1; accent-color:#e11d48; }}
  #step-label {{ font-size:10px; color:#64748b; width:45px; text-align:right; }}
  #speed-select {{ background:#f8fafc; border:1px solid #e2e8f0; color:#475569; font-size:9px; padding:2px 4px; border-radius:3px; font-family:inherit; }}

  /* LOG */
  #log-panel {{ flex:1; overflow-y:auto; padding:10px 16px; }}
  .log-entry {{ font-size:9px; margin-bottom:5px; border-left:2px solid rgba(225,29,72,0.3); padding-left:8px; }}
  .log-step {{ color:#64748b; }}
  .log-new {{ color:#94a3b8; }}

  /* PROGRESS BAR */
  #progress-section {{ padding:10px 16px; border-top:1px solid #e2e8f0; }}
  #progress-bar-bg {{ height:3px; background:#e2e8f0; border-radius:2px; overflow:hidden; margin-bottom:6px; }}
  #progress-bar {{ height:100%; background:linear-gradient(90deg,#e11d48,#fb7185); border-radius:2px; transition:width 0.4s; width:0%; }}
  #progress-text {{ font-size:9px; color:#94a3b8; display:flex; justify-content:space-between; }}

  /* LEFT PANEL SHARED */
  #left-panel {{ width:300px; border-right:1px solid #e2e8f0; display:flex; flex-direction:column; overflow:hidden; flex-shrink:0; background:#ffffff; }}
  .panel-section {{ padding:14px 16px; border-bottom:1px solid #f1f5f9; }}
  .panel-label {{ font-size:9px; letter-spacing:0.15em; color:#94a3b8; margin-bottom:10px; }}
  #search-box {{ width:100%; background:#f8fafc; border:1px solid #e2e8f0; border-radius:4px; padding:6px 10px; color:#1e293b; font-size:11px; font-family:inherit; outline:none; }}

  /* MAP */
  #map-container {{ flex:1; position:relative; }}
  #map {{ width:100%; height:100%; }}

  /* RIGHT PANEL */
  #right-panel {{ width:210px; border-left:1px solid #e2e8f0; display:flex; flex-direction:column; overflow-y:auto; padding:14px; gap:14px; flex-shrink:0; background:#ffffff; }}
  #right-panel .panel-label {{ font-size:9px; letter-spacing:0.15em; color:#94a3b8; margin-bottom:8px; }}

  #infected-count {{ font-size:32px; font-weight:bold; color:#cbd5e1; line-height:1; }}
  #infected-count.active {{ color:#e11d48; }}
  #infected-sub {{ font-size:9px; color:#94a3b8; margin-top:3px; }}

  .comm-row {{ margin-bottom:6px; }}
  .comm-header {{ display:flex; justify-content:space-between; font-size:9px; margin-bottom:2px; }}
  .comm-bar-bg {{ height:3px; background:#f1f5f9; border-radius:2px; }}
  .comm-bar {{ height:100%; border-radius:2px; transition:width 0.4s; width:0%; }}

  .risk-row {{ display:flex; justify-content:space-between; margin-bottom:5px; font-size:9px; }}

  /* TOOLTIP */
  #tooltip {{ position:fixed; background:rgba(255,255,255,0.98); border:1px solid #e2e8f0; border-radius:6px; padding:10px 13px; font-size:10px; pointer-events:none; display:none; z-index:9999; max-width:240px; box-shadow:0 4px 16px rgba(0,0,0,0.1); }}
  #tooltip .t-id {{ font-weight:bold; color:#1e293b; font-size:12px; margin-bottom:4px; }}
  #tooltip .t-name {{ color:#64748b; font-size:9px; margin-bottom:8px; }}
  #tooltip .t-grid {{ display:grid; grid-template-columns:1fr 1fr; gap:3px 10px; }}
  #tooltip .t-label {{ color:#94a3b8; }}
  #tooltip .t-value {{ color:#1e293b; text-align:right; }}
  #tooltip .t-infected {{ color:#e11d48; }}
  #tooltip .t-safe {{ color:#16a34a; }}

  /* LEGEND */
  #legend {{ position:absolute; bottom:16px; left:16px; background:rgba(255,255,255,0.96); border:1px solid #e2e8f0; border-radius:6px; padding:10px 13px; z-index:1000; font-size:9px; box-shadow:0 2px 8px rgba(0,0,0,0.08); }}
  #legend .leg-title {{ color:#94a3b8; letter-spacing:0.1em; margin-bottom:8px; }}
  .leg-row {{ display:flex; align-items:center; gap:6px; margin-bottom:4px; color:#64748b; }}
  .leg-dot {{ width:10px; height:10px; border-radius:50%; flex-shrink:0; }}
  .leg-line {{ width:20px; height:2px; flex-shrink:0; }}

  ::-webkit-scrollbar {{ width:3px; }}
  ::-webkit-scrollbar-track {{ background:transparent; }}
  ::-webkit-scrollbar-thumb {{ background:#e2e8f0; border-radius:2px; }}

  @keyframes pulse {{ 0%,100% {{ opacity:1; }} 50% {{ opacity:0.3; }} }}
  #live-dot {{ width:8px; height:8px; border-radius:50%; background:#cbd5e1; margin-right:8px; flex-shrink:0; }}
  #live-dot.running {{ background:#e11d48; box-shadow:0 0 10px rgba(225,29,72,0.5); animation:pulse 1s infinite; }}
</style>
</head>
<body>

<div id="header">
  <div style="display:flex;align-items:center;">
    <div id="live-dot"></div>
    <h1>NHS HOSPITAL NETWORK — CYBERATTACK SI PROPAGATION MODEL</h1>
  </div>
  <div id="status-bar">
    <div>STEP <span id="hdr-step">—</span></div>
    <div>INFECTED <span id="hdr-infected">0</span> / <span id="hdr-total">125</span></div>
    <div>SEED <span id="hdr-seed">—</span></div>
    <div>β = {BETA}</div>
  </div>
</div>

<div id="main">
  <!-- LEFT PANEL -->
  <div id="left-panel">
    <div class="panel-section">
      <div class="panel-label">SELECT SEED HOSPITAL(S)</div>
      <input id="search-box" type="text" placeholder="Search by name or code...">
      <div id="hospital-list"></div>
    </div>

    <div class="panel-section">
      <div class="panel-label">SELECTED SEEDS (<span id="seed-count">0</span>)</div>
      <div id="seeds-display"></div>
      <div id="btn-row">
        <button class="btn" id="btn-run" disabled>▶ RUN</button>
        <button class="btn" id="btn-reset">↺ RESET</button>
      </div>
      <div id="btn-row" style="margin-top:5px;">
        <button class="btn" id="btn-export">⬇ EXPORT HTML</button>
      </div>
      <div id="playback">
        <input type="range" id="step-slider" min="0" max="{MAX_STEPS}" value="0" disabled>
        <div id="step-label">T = 0</div>
        <select id="speed-select">
          <option value="800">Slow</option>
          <option value="500" selected>Med</option>
          <option value="250">Fast</option>
          <option value="100">×4</option>
        </select>
      </div>
    </div>

    <div class="panel-section" style="padding-bottom:8px;">
      <div class="panel-label">PROPAGATION LOG</div>
      <div id="log-panel"></div>
    </div>

    <div id="progress-section">
      <div id="progress-bar-bg"><div id="progress-bar"></div></div>
      <div id="progress-text">
        <span>NETWORK COVERAGE</span>
        <span id="progress-pct">0%</span>
      </div>
    </div>
  </div>

  <!-- MAP -->
  <div id="map-container">
    <div id="map"></div>
    <div id="legend">
      <div class="leg-title">LEGEND</div>
      <div class="leg-row"><div class="leg-dot" style="background:#e11d48;"></div> Infected</div>
      <div class="leg-row"><div class="leg-dot" style="background:white;border:2px solid #e11d48"></div> Seed hospital</div>
      <div class="leg-row"><div class="leg-dot" style="background:#94a3b8"></div> Susceptible</div>
      <div class="leg-row"><div class="leg-line" style="background:rgba(225,29,72,0.7)"></div> Active transmission</div>
      <div class="leg-row"><div class="leg-line" style="background:#cbd5e1"></div> Transfer route</div>
      <div style="margin-top:8px; color:#94a3b8; font-size:8px;">Node size = transfer volume<br>Edge weight = transfer count</div>
    </div>
  </div>

  <!-- RIGHT PANEL -->
  <div id="right-panel">
    <div>
      <div class="panel-label">INFECTION STATUS</div>
      <div id="infected-count">0</div>
      <div id="infected-sub">hospitals infected</div>
      <div style="margin-top:6px;font-size:9px;color:#64748b;">
        <span id="susceptible-count" style="color:#16a34a">125</span> susceptible
      </div>
    </div>

    <div>
      <div class="panel-label">BY COMMUNITY</div>
      <div id="community-bars"></div>
    </div>

    <div>
      <div class="panel-label">HIGH RISK — UNINFECTED</div>
      <div id="risk-list"></div>
    </div>

    <div style="margin-top:auto;">
      <div class="panel-label">SI MODEL PARAMS</div>
      <div style="font-size:9px;color:#94a3b8;line-height:1.7;">
        β = {BETA} base rate<br>
        Prob = 1 − e^(−β·w/w_max)<br>
        Bidirectional × 0.6<br>
        No recovery (SI)<br>
        {N_RUNS} runs per seed<br>
        Median step shown
      </div>
    </div>
  </div>
</div>

<div id="tooltip">
  <div class="t-id" id="tt-id"></div>
  <div class="t-name" id="tt-name"></div>
  <div class="t-grid">
    <div class="t-label">Community</div><div class="t-value" id="tt-comm"></div>
    <div class="t-label">In-degree</div><div class="t-value" id="tt-indeg"></div>
    <div class="t-label">Bridges</div><div class="t-value" id="tt-bridges"></div>
    <div class="t-label">Infect freq</div><div class="t-value" id="tt-freq"></div>
    <div class="t-label">Epidemic %</div><div class="t-value" id="tt-epi"></div>
    <div class="t-label">Median step</div><div class="t-value" id="tt-mstep"></div>
    <div class="t-label">Status</div><div class="t-value" id="tt-status"></div>
  </div>
</div>

<script>
// ── DATA ──────────────────────────────────────────────────────
const HOSPITALS    = {hospitals_json};
const EDGES        = {edges_json};
const SIM_RESULTS  = {sim_results_json};
const TIMELINES    = {timelines_json};
const COLOR_MAP    = {color_map_json};
const COMMUNITY_IDS = {community_ids_json};
const MAX_STEPS    = {MAX_STEPS};
const BETA         = {BETA};
const MAX_IN_DEGREE = {max_in_degree};
const MAX_TRANSFERS = {max_transfers};

// ── MAP INIT ──────────────────────────────────────────────────
const map = L.map('map', {{
  center: [52.8, -1.5],
  zoom: 6,
  zoomControl: true,
  attributionControl: false
}});

L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  maxZoom: 18, subdomains: 'abcd'
}}).addTo(map);

// ── ICB BOUNDARIES ────────────────────────────────────────────
const icbGeo = {icb_geo_json};
const icbLayer = L.geoJSON(icbGeo, {{
  style: {{
    color: 'rgba(100,116,139,0.4)',
    weight: 0.8,
    fillColor: 'rgba(100,116,139,0.04)',
    fillOpacity: 1
  }},
  onEachFeature: (feature, layer) => {{
    const name = feature.properties.icb_name || feature.properties.ICB23NM || '';
    layer.bindTooltip(name, {{
      permanent: false, sticky: true,
      className: 'icb-tooltip',
      direction: 'top'
    }});
  }}
}}).addTo(map);

// ── BUILD HOSPITAL LOOKUP ─────────────────────────────────────
const hospMap = {{}};
HOSPITALS.forEach(h => hospMap[h.id] = h);

// ── DRAW EDGES ────────────────────────────────────────────────
const MAX_WEIGHT = Math.max(...EDGES.map(e => e.weight));
const edgeLayerGroup = L.layerGroup().addTo(map);
const edgeLines = {{}};  // "src-tgt" -> polyline

function buildEdgeLines() {{
  edgeLayerGroup.clearLayers();
  EDGES.forEach(edge => {{
    const s = hospMap[edge.source];
    const t = hospMap[edge.target];
    if (!s || !t) return;
    const opacity = 0.06 + (edge.weight / MAX_WEIGHT) * 0.15;
    const width   = 0.4 + (edge.weight / MAX_WEIGHT) * 1.6;
    const line = L.polyline([[s.lat, s.lon], [t.lat, t.lon]], {{
      color: '#94a3b8', weight: width, opacity: opacity
    }});
    edgeLayerGroup.addLayer(line);
    edgeLines[`${{edge.source}}-${{edge.target}}`] = line;
  }});
}}
buildEdgeLines();

// ── DRAW HOSPITAL MARKERS ─────────────────────────────────────
const markerGroup = L.layerGroup().addTo(map);
const markers = {{}};  // id -> circleMarker

const tooltip = document.getElementById('tooltip');

function getRadius(h) {{
  const base = 4;
  const scale = (h.weighted_in_degree / MAX_IN_DEGREE);
  return base + scale * 14;
}}

function buildMarkers() {{
  markerGroup.clearLayers();
  HOSPITALS.forEach(h => {{
    const r = getRadius(h);
    const marker = L.circleMarker([h.lat, h.lon], {{
      radius: r,
      fillColor: h.color,
      color: 'rgba(30,41,59,0.2)',
      weight: 0.8,
      fillOpacity: 0.82
    }}).addTo(markerGroup);

    marker.on('mouseover', (e) => showTooltip(e, h));
    marker.on('mousemove', (e) => moveTooltip(e));
    marker.on('mouseout', () => {{ tooltip.style.display = 'none'; }});
    marker.on('click', () => toggleSeed(h.id));
    markers[h.id] = marker;
  }});
}}
buildMarkers();

function showTooltip(e, h) {{
  const isInfected = infectedSet.has(h.id);
  const isSeed = seeds.includes(h.id);
  document.getElementById('tt-id').textContent = h.id + (isSeed ? ' ★ SEED' : '');
  document.getElementById('tt-name').textContent = h.name;
  document.getElementById('tt-comm').textContent = h.community_id;
  document.getElementById('tt-indeg').textContent = h.weighted_in_degree?.toLocaleString() || '—';
  document.getElementById('tt-bridges').textContent = h.cross_community_bridges || '—';
  document.getElementById('tt-freq').textContent = h.infection_frequency != null ? (h.infection_frequency * 100).toFixed(1) + '%' : '—';
  document.getElementById('tt-epi').textContent = h.mean_epidemic_size_pct != null ? h.mean_epidemic_size_pct.toFixed(1) + '%' : '—';
  document.getElementById('tt-mstep').textContent = h.mean_step_to_infection != null ? h.mean_step_to_infection.toFixed(1) : '—';
  const statusEl = document.getElementById('tt-status');
  statusEl.textContent = isInfected ? 'INFECTED' : 'SUSCEPTIBLE';
  statusEl.className = isInfected ? 't-value t-infected' : 't-value t-safe';
  tooltip.style.display = 'block';
  moveTooltip(e);
}}

function moveTooltip(e) {{
  const me = e.originalEvent || e;
  tooltip.style.left = (me.clientX + 14) + 'px';
  tooltip.style.top  = (me.clientY - 10) + 'px';
}}

// ── STATE ─────────────────────────────────────────────────────
let seeds = [];
let infectedSet = new Set();
let currentStep = 0;
let animTimer = null;
let isRunning = false;

// ── HOSPITAL LIST UI ──────────────────────────────────────────
const hospList = document.getElementById('hospital-list');
const searchBox = document.getElementById('search-box');

function renderHospList(filter = '') {{
  hospList.innerHTML = '';
  const fl = filter.toLowerCase();
  HOSPITALS
    .filter(h => !fl || h.id.toLowerCase().includes(fl) || h.name.toLowerCase().includes(fl))
    .sort((a, b) => b.weighted_in_degree - a.weighted_in_degree)
    .forEach(h => {{
      const div = document.createElement('div');
      div.className = 'hosp-item' + (seeds.includes(h.id) ? ' selected' : '');
      div.innerHTML = `
        <div class="hosp-dot" style="background:${{h.color}}"></div>
        <div class="hosp-item-text">
          <div class="hosp-id">${{h.id}}</div>
          <div class="hosp-name">${{h.name}}</div>
        </div>
        <div class="hosp-risk">f=${{h.infection_frequency != null ? (h.infection_frequency*100).toFixed(0)+'%' : '—'}}</div>
      `;
      div.onclick = () => toggleSeed(h.id);
      hospList.appendChild(div);
    }});
}}
renderHospList();
searchBox.addEventListener('input', () => renderHospList(searchBox.value));

// ── SEED MANAGEMENT ───────────────────────────────────────────
function toggleSeed(id) {{
  if (isRunning) return;
  if (seeds.includes(id)) {{
    seeds = seeds.filter(s => s !== id);
  }} else {{
    seeds.push(id);
  }}
  renderSeedTags();
  renderHospList(searchBox.value);
  updateSeedMarkers();
  document.getElementById('btn-run').disabled = seeds.length === 0;
}}

function renderSeedTags() {{
  const el = document.getElementById('seeds-display');
  el.innerHTML = seeds.map(s => `<span class="seed-tag" onclick="toggleSeed('${{s}}')">${{s}} ×</span>`).join('');
  document.getElementById('seed-count').textContent = seeds.length;
}}

function updateSeedMarkers() {{
  HOSPITALS.forEach(h => {{
    if (!markers[h.id]) return;
    if (seeds.includes(h.id)) {{
      markers[h.id].setStyle({{ color: '#1e293b', weight: 2.5, fillColor: h.color }});
    }} else {{
      markers[h.id].setStyle({{ color: 'rgba(30,41,59,0.2)', weight: 0.8 }});
    }}
  }});
}}

// ── SIMULATION ────────────────────────────────────────────────
// Multi-seed: merge timelines from all seeds, take min step per hospital
function getMergedTimeline() {{
  const merged = {{}};   // hosp_id -> min median_step across seeds
  seeds.forEach(seed => {{
    const tl = TIMELINES[seed];
    if (!tl) return;
    // Find median_step for each hospital from sim results
    const sr = SIM_RESULTS[seed];
    if (!sr) return;
    Object.entries(sr).forEach(([hid, stats]) => {{
      if (stats.median_step == null) return;
      if (merged[hid] == null || stats.median_step < merged[hid]) {{
        merged[hid] = stats.median_step;
      }}
    }});
    // Always include seeds at step 0
    seeds.forEach(s => {{ merged[s] = 0; }});
  }});
  return merged;
}}

let mergedTimeline = {{}};  // hosp_id -> step_infected

function startSim() {{
  if (seeds.length === 0) return;
  resetSim(false);
  mergedTimeline = getMergedTimeline();
  isRunning = true;
  document.getElementById('live-dot').className = 'running';
  document.getElementById('btn-run').disabled = true;
  document.getElementById('step-slider').disabled = false;
  document.getElementById('hdr-seed').textContent = seeds.join(', ');
  addLog(0, seeds, []);
  applyStep(0);
  runNextStep();
}}

function runNextStep() {{
  const speed = parseInt(document.getElementById('speed-select').value);
  animTimer = setTimeout(() => {{
    if (currentStep >= MAX_STEPS || infectedSet.size === HOSPITALS.length) {{
      stopSim();
      return;
    }}
    currentStep++;
    document.getElementById('step-slider').value = currentStep;
    applyStep(currentStep);
    runNextStep();
  }}, speed);
}}

function applyStep(step) {{
  const newlyInfected = [];

  // Determine infected set at this step
  Object.entries(mergedTimeline).forEach(([hid, infStep]) => {{
    if (infStep <= step && !infectedSet.has(hid)) {{
      infectedSet.add(hid);
      newlyInfected.push(hid);
    }}
  }});

  updateMap(newlyInfected);
  updateStats();
  document.getElementById('hdr-step').textContent = step;
  document.getElementById('step-label').textContent = `T = ${{step}}`;
  if (newlyInfected.length > 0) addLog(step, [], newlyInfected);
}}

function updateMap(newlyInfected) {{
  // Flash active edges for newly infected
  newlyInfected.forEach(hid => {{
    EDGES.forEach(edge => {{
      const key1 = `${{edge.source}}-${{edge.target}}`;
      const key2 = `${{edge.target}}-${{edge.source}}`;
      if ((edge.source === hid || edge.target === hid) && infectedSet.has(edge.source) && infectedSet.has(edge.target)) {{
        const line = edgeLines[key1] || edgeLines[key2];
        if (line) {{
          line.setStyle({{ color: '#e11d48', opacity: 0.7, weight: 2 }});
          setTimeout(() => {{
            line.setStyle({{ color: '#e11d48', opacity: 0.35, weight: 1.2 }});
          }}, 500);
        }}
      }}
    }});
  }});

  // Update all markers
  HOSPITALS.forEach(h => {{
    const marker = markers[h.id];
    if (!marker) return;
    const isInfected = infectedSet.has(h.id);
    const isSeed = seeds.includes(h.id);
    if (isInfected) {{
      marker.setStyle({{
        fillColor: '#e11d48',
        color: isSeed ? '#1e293b' : '#fb7185',
        weight: isSeed ? 2.5 : 1.2,
        fillOpacity: 0.95
      }});
    }} else {{
      marker.setStyle({{
        fillColor: h.color,
        color: isSeed ? '#1e293b' : 'rgba(30,41,59,0.2)',
        weight: isSeed ? 2.5 : 0.8,
        fillOpacity: 0.82
      }});
    }}
  }});
}}

function updateStats() {{
  const total = HOSPITALS.length;
  const count = infectedSet.size;
  const pct   = Math.round((count / total) * 100);

  document.getElementById('infected-count').textContent = count;
  document.getElementById('infected-count').className = count > 0 ? 'active' : '';
  document.getElementById('hdr-infected').textContent = count;
  document.getElementById('susceptible-count').textContent = total - count;
  document.getElementById('progress-bar').style.width = pct + '%';
  document.getElementById('progress-pct').textContent = pct + '%';

  // Community bars
  const commBars = document.getElementById('community-bars');
  commBars.innerHTML = COMMUNITY_IDS.map(cid => {{
    const hosps = HOSPITALS.filter(h => h.community_id === cid);
    const inf   = hosps.filter(h => infectedSet.has(h.id)).length;
    const p     = hosps.length > 0 ? Math.round((inf/hosps.length)*100) : 0;
    const color = COLOR_MAP[cid] || '#94a3b8';
    return `<div class="comm-row">
      <div class="comm-header">
        <span style="color:#64748b">C${{cid}}</span>
        <span style="color:${{inf > 0 ? '#e11d48' : '#94a3b8'}}">${{inf}}/${{hosps.length}}</span>
      </div>
      <div class="comm-bar-bg"><div class="comm-bar" style="width:${{p}}%;background:${{color}}"></div></div>
    </div>`;
  }}).join('');

  // High risk uninfected
  const riskList = document.getElementById('risk-list');
  const uninfected = HOSPITALS
    .filter(h => !infectedSet.has(h.id) && h.infection_frequency != null)
    .sort((a, b) => b.infection_frequency - a.infection_frequency)
    .slice(0, 8);
  riskList.innerHTML = uninfected.length === 0
    ? '<div style="font-size:9px;color:#94a3b8;font-style:italic">All critical nodes infected</div>'
    : uninfected.map(h => `
      <div class="risk-row">
        <span style="color:#94a3b8">${{h.id}}</span>
        <span style="color:#f59e0b">${{(h.infection_frequency*100).toFixed(0)}}%</span>
      </div>`).join('');
}}

function stopSim() {{
  clearTimeout(animTimer);
  isRunning = false;
  document.getElementById('live-dot').className = '';
  document.getElementById('btn-run').disabled = false;
}}

function resetSim(fullReset = true) {{
  stopSim();
  infectedSet.clear();
  currentStep = 0;
  mergedTimeline = {{}};
  document.getElementById('hdr-step').textContent = '—';
  document.getElementById('step-slider').value = 0;
  document.getElementById('step-slider').disabled = true;
  document.getElementById('log-panel').innerHTML = '';
  if (fullReset) {{
    seeds = [];
    renderSeedTags();
    renderHospList(searchBox.value);
    document.getElementById('hdr-seed').textContent = '—';
    document.getElementById('btn-run').disabled = true;
  }}
  // Reset all markers
  HOSPITALS.forEach(h => {{
    if (!markers[h.id]) return;
    markers[h.id].setStyle({{
      fillColor: h.color,
      color: 'rgba(30,41,59,0.2)',
      weight: 0.8,
      fillOpacity: 0.82
    }});
  }});
  // Reset all edges
  EDGES.forEach(edge => {{
    const line = edgeLines[`${{edge.source}}-${{edge.target}}`];
    if (!line) return;
    const opacity = 0.06 + (edge.weight / MAX_WEIGHT) * 0.15;
    const width   = 0.4 + (edge.weight / MAX_WEIGHT) * 1.6;
    line.setStyle({{ color: '#94a3b8', weight: width, opacity: opacity }});
  }});
  updateStats();
  document.getElementById('infected-count').className = '';
}}

// ── LOG ───────────────────────────────────────────────────────
function addLog(step, seeds, newly) {{
  const panel = document.getElementById('log-panel');
  const div = document.createElement('div');
  div.className = 'log-entry';
  const preview = newly.slice(0, 4).join(', ') + (newly.length > 4 ? ` +${{newly.length - 4}} more` : '');
  div.innerHTML = `
    <div class="log-step">T=${{step}} — ${{infectedSet.size}} infected (${{Math.round(infectedSet.size/HOSPITALS.length*100)}}%)</div>
    ${{newly.length > 0 ? `<div class="log-new">+${{preview}}</div>` : ''}}
    ${{seeds.length > 0 ? `<div class="log-new">Seeds: ${{seeds.join(', ')}}</div>` : ''}}
  `;
  panel.insertBefore(div, panel.firstChild);
}}

// ── SLIDER ────────────────────────────────────────────────────
document.getElementById('step-slider').addEventListener('input', function() {{
  if (isRunning) return;
  const targetStep = parseInt(this.value);
  // Rewind: rebuild infected set from scratch
  infectedSet.clear();
  currentStep = targetStep;
  Object.entries(mergedTimeline).forEach(([hid, infStep]) => {{
    if (infStep <= targetStep) infectedSet.add(hid);
  }});
  updateMap([]);
  updateStats();
  document.getElementById('hdr-step').textContent = targetStep;
  document.getElementById('step-label').textContent = `T = ${{targetStep}}`;
}});

// ── BUTTONS ───────────────────────────────────────────────────
document.getElementById('btn-run').addEventListener('click', startSim);
document.getElementById('btn-reset').addEventListener('click', () => resetSim(true));

document.getElementById('btn-export').addEventListener('click', () => {{
  // Save current page as HTML
  const blob = new Blob([document.documentElement.outerHTML], {{type: 'text/html'}});
  const a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = 'hospital_si_snapshot.html';
  a.click();
}});

// ── COMMUNITY LEGEND (right panel) ───────────────────────────
const commBarsInit = document.getElementById('community-bars');
commBarsInit.innerHTML = COMMUNITY_IDS.map(cid => {{
  const hosps = HOSPITALS.filter(h => h.community_id === cid);
  const color = COLOR_MAP[cid] || '#94a3b8';
  return `<div class="comm-row">
    <div class="comm-header">
      <span style="color:#64748b">C${{cid}} — ${{hosps.length}} hospitals</span>
      <span style="color:#94a3b8">0/${{hosps.length}}</span>
    </div>
    <div class="comm-bar-bg"><div class="comm-bar" style="width:0%;background:${{color}}"></div></div>
  </div>`;
}}).join('');

document.getElementById('hdr-total').textContent = HOSPITALS.length;
</script>
</body>
</html>
"""


# ── 8. WRITE OUTPUT

In [19]:
OUT_PATH = "hospital_si_map.html"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(HTML)

print(f"\n✓ Map saved to: {OUT_PATH}")
print(f"  Opening in browser...")
webbrowser.open(f"file://{'/Users/swethamavuru/Desktop/Kunal work/OPR-CAN/Coding/iReACH/HES_data/hospital_si_map.html'}")


✓ Map saved to: hospital_si_map.html
  Opening in browser...


True